# The Cox model with Breslow's ties, and its predicted survival

When several patients die on the same day, the Cox partial likelihood needs a rule for the tie. Survival.jl uses Efron's (1977); much of the published literature, and the SAS, BMDP and early S software it was computed with, used Breslow's (1974), so its coefficients cannot be reproduced with Survival.jl. Nor does Survival.jl predict a patient's survival curve from a fitted Cox model. Both are written out here: `cox` fits the model with Breslow's ties, and `survival` gives the predicted survival S(t, R) = S₀(t)^exp(R − c) with Kalbfleisch and Prentice's baseline. They are tested against the Mayo Clinic model for primary biliary cirrhosis (Dickson et al. 1989): both columns of its Table 2 and the seven years of its Table 4.

The data is `pbc`, from PollisDatasets: the 418 Mayo patients as Fleming and Harrington (1991) published them.

In [ ]:
Pollis.packages("Survival", "ForwardDiff", "Distributions", "PollisDatasets", "Plots")

## Set-up

`replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share. The follow-up `days`, the event `dead`, the 312 trial patients who built the model, and `riskfactors`, the five measurements of the Mayo model: log bilirubin, log albumin, age, log prothrombin time and edema.

In [ ]:
using Survival, ForwardDiff, Distributions, PollisDatasets, Plots, Printf, Statistics, LinearAlgebra

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

pbc = dataset("pbc");  # one row per patient
days = Float64.(pbc.time);  # from entry to death, transplantation or the end of follow-up
dead = pbc.status .== 2;  # death; transplanted patients are censored at transplantation
trial = 1:312;  # the two Mayo trials: they built the model
names5 = ["log bilirubin", "log albumin", "age", "log prothrombin time", "edema"];  # the order of Table 2

"""The five measurements of the Mayo model for the patients in `rows`, one column each, in the order of Table 2."""
riskfactors(rows; protime=pbc.protime) = Float64.(hcat(log.(pbc.bili[rows]), log.(pbc.albumin[rows]), pbc.age[rows], log.(protime[rows]), pbc.edema[rows]))

## The method

### Breslow's ties

Breslow's partial likelihood lets all patients who die on the same day share one risk set: each death contributes η_i − log Σ_{j at risk} exp(η_j), with η = Xβ. `cox` maximises it by Newton's method, with exact derivatives from ForwardDiff, and returns the coefficients, standard errors from the inverse of the information, Wald chi-squares and p values, and the likelihood ratio chi-square of the whole model.

In [ ]:
"""
	partial(β, X, t, e)

Breslow's (1974) partial log likelihood of the Cox model with covariates `X`, times `t` and events `e`: the patients who die on the same day
share one risk set.
"""
function partial(β, X, t, e)
	η = X * β
	sum(η[i] - log(sum(exp.(η[t .>= t[i]]))) for i in findall(e))
end

"""
	cox(X, t, e)

Fit the Cox model by Newton's method on Breslow's partial likelihood: coefficients, standard errors, Wald chi-squares and their p values,
and the likelihood ratio chi-square of the whole model.
"""
function cox(X, t, e)
	f(β) = partial(β, X, t, e)
	β = zeros(size(X, 2))
	for _ in 1:25
		β -= ForwardDiff.hessian(f, β) \ ForwardDiff.gradient(f, β)
	end
	se = sqrt.(diag(inv(-ForwardDiff.hessian(f, β))))
	chi2 = (β ./ se) .^ 2
	(β = β, se = se, chi2 = chi2, p = ccdf.(Chisq(1), chi2), LR = 2 * (f(β) - f(zero(β))), X = X, t = t, e = e)
end

### Predicted survival

A patient with risk score R = Xβ has predicted survival S(t, R) = S₀(t)^exp(R − c), where c is the mean risk score and S₀ the survival of a patient at c. Kalbfleisch and Prentice's (1980) estimate of S₀ is a product, over the days with deaths, of factors α, each the root of Σ_deaths w / (1 − α^w) = Σ_at risk w, with w = exp(R − c); `survival` finds each root by bisection.

In [ ]:
"""
	survival(m)

The predicted survival S(t, R) = S0(t)^exp(R - c) of a model fitted by `cox`, at t days for a patient with risk score R, where c is the
mean risk score and S0 is Kalbfleisch and Prentice's (1980) estimate.
"""
function survival(m)
	R = m.X * m.β
	c = mean(R)
	w = exp.(R .- c)
	times = sort(unique(m.t[m.e]))
	α = map(times) do s
		deaths = findall(m.e .& (m.t .== s))
		g(a) = sum(w[i] / (1 - a^w[i]) for i in deaths) - sum(w[m.t .>= s])  # increasing in a
		lo, hi = 0.0, 1.0
		for _ in 1:100
			mid = (lo + hi) / 2
			g(mid) > 0 ? (hi = mid) : (lo = mid)
		end
		(lo + hi) / 2
	end
	S0 = cumprod(α)
	(t, r) -> (j = searchsortedlast(times, t); j == 0 ? 1.0 : S0[j]^exp(r - c))
end

## Tests

### Without ties, Breslow is Cox

When no two deaths share a day, Breslow's, Efron's and every other rule give the same partial likelihood, so `cox` must agree with Survival.jl. Breaking the ties of the trial patients by a few minutes each gives such data:

In [ ]:
X = riskfactors(trial);
untied = days[trial] .+ (1:312) ./ 10_000;  # distinct times, the order of tied deaths by row
mine, theirs = cox(X, untied, dead[trial]), fit(CoxModel, X, EventTime.(untied, dead[trial]))
@printf("%-24s%16s%16s%12s\n", "", "cox", "Survival.jl", "difference")
for (name, a, b) in zip(names5, mine.β, coef(theirs))
	@printf("%-24s%16.10f%16.10f%12.1e\n", name, a, b, a - b)
end

### Table 2: the Mayo model

The paper's fits were made with SAS PHGLM (p. 3), with Breslow's ties. The initial model is fitted to the 312 trial patients; the final one to all 418, after estimating the two missing prothrombin times by a regression on the other four measurements, as the paper estimated its missing values (p. 3). Survival.jl's Efron fit is printed first, to show the difference the ties make.

In [ ]:
"""
	table2(title, m, published)

Replicate a column of Table 2: `published` holds the coefficient, standard error, chi-square and p value of each variable, as printed,
with `nothing` for a p value printed as < 0.0001.
"""
function table2(title, m, published)
	rows = []
	for (i, (b, se, chi2, p)) in enumerate(published)
		push!(rows, (names5[i], m.β[i], b), ("  se", m.se[i], se), ("  chi-square", m.chi2[i], chi2))
		p === nothing ? push!(rows, ("  p < 0.0001", m.p[i] < 0.0001, "1")) : push!(rows, ("  p", m.p[i], p))
	end
	replicate(title, rows)
end

efron = fit(CoxModel, X, EventTime.(days[trial], dead[trial]))
replicate("Survival.jl, Efron's ties (Table 2, initial model, p. 4)", collect(zip(names5, coef(efron), ["0.8792", "-3.053", "0.0333", "3.016", "0.7847"])))

initial = cox(X, days[trial], dead[trial]);
table2("cox, Breslow's ties: initial model, 312 patients (Table 2, p. 4)", initial, [("0.8792", "0.0987", "79.30", nothing), ("-3.053", "0.724", "17.78", nothing),
	("0.0333", "0.0087", "14.76", "0.0001"), ("3.016", "1.024", "8.68", "0.0032"), ("0.7847", "0.2991", "6.88", "0.0087")])
replicate("The whole model (p. 4)", [("likelihood ratio chi-square", initial.LR, "199.2")])

missingpt = findall(ismissing, pbc.protime)  # two patients
known = setdiff(1:418, missingpt);
Z(rows) = Float64.(hcat(ones(length(rows)), log.(pbc.bili[rows]), log.(pbc.albumin[rows]), pbc.age[rows], pbc.edema[rows]))
b = Z(known) \ log.(Float64.(pbc.protime[known]));
protime = Float64.(coalesce.(pbc.protime, NaN));
protime[missingpt] = exp.(Z(missingpt) * b);
Xall = riskfactors(1:418; protime=protime);
final = cox(Xall, days, dead);
table2("cox, Breslow's ties: final model, 418 patients (Table 2, p. 4)", final, [("0.8707", "0.0826", "111.03", nothing), ("-2.533", "0.648", "15.27", "0.0001"),
	("0.0394", "0.0077", "26.53", nothing), ("2.380", "0.767", "9.64", "0.0019"), ("0.8592", "0.2711", "10.04", "0.0015")])

### Table 4: the baseline survival

Table 4 (p. 6) gives S₀(t) of the final model for years 1 to 7, at the mean risk score of the 418 patients.

In [ ]:
S = survival(final);
c = mean(Xall * final.β);
replicate("survival: baseline S0(t), final model (Table 4, p. 6)", [("year $y", S(y * 365.25, c), p) for (y, p) in
	zip(1:7, ["0.970", "0.941", "0.883", "0.833", "0.774", "0.721", "0.651"])])

## Example

The paper's two worked patients (p. 6): a 52-year-old with bilirubin 0.5, albumin 4.5, prothrombin time 10.1 and no edema, and one with bilirubin 13.9, albumin 2.8, prothrombin time 13.8 and edema treated with diuretics. Their predicted survival curves from the final model; the paper gives the first a 96% chance of surviving 5 years and the second a 43% chance of surviving 1 year.

In [ ]:
patients = [("patient 1", [log(0.5), log(4.5), 52, log(10.1), 0]), ("patient 2", [log(13.9), log(2.8), 52, log(13.8), 0.5])];
grid = 0:15:7 * 365.25
plot(xlabel="years", ylabel="predicted survival", ylims=(0, 1), size=(560, 330))
for (label, x) in patients
	r = dot(x, final.β)
	@printf("%s: R = %.2f, 1-year survival %.2f, 5-year survival %.2f\n", label, r, S(365.25, r), S(5 * 365.25, r))
	plot!(grid ./ 365.25, S.(grid, r), linetype=:steppost, label=label)
end
plot!()

## What differs

Without ties, `cox` and Survival.jl agree to 6 decimals or more, the difference being Survival.jl's stopping tolerance. On the Mayo data, all 20 values of the initial model's column of Table 2 and all seven years of Table 4 agree to the published precision. The rest:

- **The likelihood ratio chi-square (199.13 against 199.2).** The coefficients and standard errors it comes from all agree. Not explained.
- **The final model, in the last published digit** (log bilirubin 0.8706 against 0.8707, log prothrombin time 2.379 against 2.380, and four chi-squares by 0.01 or 0.02). The paper estimated missing values for the 16 declined patients who lacked prothrombin time, cirrhosis or cholestasis, with regressions fitted to the 90 complete patients (p. 3). The public file lacks only two prothrombin times, so the regression here is simpler; setting those two times to anything from 9 to 13 seconds moves the coefficients in the third decimal, gaps of this size. The initial model, which needs no estimated values, agrees throughout.

Survival.jl's Efron fit, by contrast, agrees with Table 2 to only 1 to 3 decimals: the ties are the whole difference.

**Not included: the one-sample log-rank test** that the paper used to check the model on the 106 patients it was not fitted to (p. 5). Its groups were formed with a biopsy variable that is not in the public data, and their sizes are not given, so its p values cannot be rebuilt: without expected results it cannot be tested.

## References

- Breslow, N. (1974). Covariance analysis of censored survival data. *Biometrics* 30(1), 89-99.
- Dickson, E. R., Grambsch, P. M., Fleming, T. R., Fisher, L. D. and Langworthy, A. (1989). Prognosis in primary biliary cirrhosis: model for decision making. *Hepatology* 10(1), 1-7. doi:10.1002/hep.1840100102
- Efron, B. (1977). The efficiency of Cox's likelihood function for censored data. *Journal of the American Statistical Association* 72(359), 557-565.
- Fleming, T. R. and Harrington, D. P. (1991). *Counting Processes and Survival Analysis*. Wiley. Appendix D: the data.
- Kalbfleisch, J. D. and Prentice, R. L. (1980). *The Statistical Analysis of Failure Time Data*. Wiley.